In [118]:
!pip install "granite-tsfm[notebooks] @ git+https://github.com/ibm-granite/granite-tsfm.git@v0.2.22"
!pip install numpy==1.25.2
!pip install --upgrade --force-reinstall pandas

  Cloning https://github.com/ibm-granite/granite-tsfm.git (to revision v0.2.22) to /tmp/pip-install-wg_85bk9/granite-tsfm_a6ca9d8091b6489ab48032dfde933ccb
  Running command git clone --filter=blob:none --quiet https://github.com/ibm-granite/granite-tsfm.git /tmp/pip-install-wg_85bk9/granite-tsfm_a6ca9d8091b6489ab48032dfde933ccb
  Running command git checkout -q 216850d0cb073e31689049c1334f701fe11bc2c3
  Resolved https://github.com/ibm-granite/granite-tsfm.git to commit 216850d0cb073e31689049c1334f701fe11bc2c3
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached numpy-1.26.4-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (61 kB)
Using cached numpy-1.26.4-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (18.3 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 2.2.4
    Uninstalling numpy-2.2.4:
      Successfully uninstalled numpy-2.2.4
ERR

  Using cached pandas-2.2.3-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (89 kB)
  Using cached numpy-2.2.4-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (62 kB)
  Using cached python_dateutil-2.9.0.post0-py2.py3-none-any.whl.metadata (8.4 kB)
  Using cached pytz-2025.2-py2.py3-none-any.whl.metadata (22 kB)
  Using cached tzdata-2025.2-py2.py3-none-any.whl.metadata (1.4 kB)
  Using cached six-1.17.0-py2.py3-none-any.whl.metadata (1.7 kB)
Using cached pandas-2.2.3-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (13.1 MB)
Using cached numpy-2.2.4-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (16.4 MB)
Using cached python_dateutil-2.9.0.post0-py2.py3-none-any.whl (229 kB)
Using cached pytz-2025.2-py2.py3-none-any.whl (509 kB)
Using cached tzdata-2025.2-py2.py3-none-any.whl (347 kB)
Using cached six-1.17.0-py2.py3-none-any.whl (11 kB)
  Attempting uninstall: pytz
    Found existing installation: pytz 2025.2
    Uninstalli

In [1]:
import os
import tempfile

from transformers import Trainer, TrainingArguments, set_seed

from tsfm_public import TinyTimeMixerForPrediction, load_dataset
from tsfm_public.toolkit import RecursivePredictor, RecursivePredictorConfig
from tsfm_public.toolkit.visualization import plot_predictions
from tsfm_public import TimeSeriesPreprocessor, get_datasets
import pandas as pd

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
import glob

In [7]:
import numpy as np
def MG_generate_interp(gamma=0.1,beta=0.2,tau=60,theta=1,n=10,x0=0.2,N=1000000,delta=0.01,interval=1):
  def MG_eq (x,x_pre):
    return x_pre * beta * (theta**n)/(theta**n+x_pre**n)-gamma*x

  def MG_rk4(x,x_pre,x_pre_f):
    interplot = (x_pre+x_pre_f)/2
    k1 = MG_eq(x,x_pre)
    k2 = MG_eq(x+delta*k1/2,interplot)
    k3 = MG_eq(x+delta*k2/2,interplot)
    k4 = MG_eq(x+delta*k3,x_pre_f)
    return x+delta*(k1+2*k2+2*k3+k4)/6

  past_len = int(np.floor(tau/delta))
  x_past = np.zeros(past_len+N+1)+0.1
  x = x0
  X = np.zeros(N+1)
  T = np.zeros(N+1)
  time = 0

  for i in range(N+1):
    X[i] = x
    x_pre = x_past[i]
    x_pre_f = x_past[i+1]
    x_delta = MG_rk4(x=x,x_pre=x_pre,x_pre_f = x_pre_f)
    x_past[i+past_len] = x_delta
    T[i] = time
    time += delta
    x = x_delta

  return T,X

In [32]:
import torch
from copy import deepcopy

def update_past_values_with_predictions(dset_test, predictions, context_length=512):
  from copy import deepcopy
  import torch

  samples = []


  for i in range(len(dset_test)):
    sample = deepcopy(dset_test[i])
    old_past = sample['past_values']        # shape [512, 1]
    new_vals_np = predictions[i]                     # shape [96, 1] expected
    #print(f"[{i}] pred shape: {new_vals_np.shape}, old tail: {old_past[-1]}")

    new_vals = torch.tensor(new_vals_np, dtype=old_past.dtype)

    combined = torch.cat([old_past, new_vals], dim=0)[-context_length:]

    sample["past_values"] = combined
    samples.append(sample)
  #dset_updated = ForecastDFDataset(data = samples,)
  #print(type(dset_updated))

  return samples

In [150]:
def perturb_last_element(samples_test,perturbation = 0.05):
  samples_test_pt = []
  for i in range(len(samples_test)):
    sample = deepcopy(samples_test[i])
    sample['past_values'][-1] += perturbation
    samples_test_pt.append(sample)
  return samples_test_pt

In [160]:
def rolling_prediction(tau,frac,location):
  df.loc[len(df)] = [None,None,None,tau,frac,location]
  # fetch the model
  model_path = f"./drive/MyDrive/ColabBackup/MG_x0_1_tau_{tau}_{frac}_{location}/output"
  matching_dirs = glob.glob(os.path.join(model_path, "checkpoint-*"))
  checkpoint_path = matching_dirs[0]
  model = TinyTimeMixerForPrediction.from_pretrained(checkpoint_path)
  # generate data
  T,X = MG_generate_interp(tau=tau,x0=1,N = 1200000)
  True_pred = np.array([X[::100][9000+i:9000+1632+i] for i in range(906)])
  df.loc[len(df)-1,'True'] = [True_pred]

  # data preprocessing
  MGdata = pd.DataFrame({
      'time': [round(x) for x in T[::100]],
      'P': X[::100],
  })
  timestamp_column_MG = "time"
  id_columns_MG = []  # mention the ids that uniquely identify a time-series.

  target_columns_MG = ["P"]
  split_config_MG = {
      "train": [0, 6000],
      "valid": [6000, 9000],
      "test": [
          9000,
          10001,
      ],
  }
  column_specifiers_MG = {
      "timestamp_column": timestamp_column_MG,
      "id_columns": id_columns_MG,
      "target_columns": target_columns_MG,
      "control_columns": [],
  }
  tsp = TimeSeriesPreprocessor(
      **column_specifiers_MG,
      context_length=512,
      prediction_length=96,                          # match your time interval
      scaling=False,
      encode_categorical=False,
      #scaler_type="standard",           # match training
  )

  # manually scaler

  scale_mean = np.zeros(906)
  scale_sd = np.zeros(906)

  for i in range (906):
    scale_mean[i] = np.mean(X[::100][9000+i:9000+i+96])
    scale_sd[i] = np.std(X[::100][9000+i:9000+i+96])

  _, _, dset_test = get_datasets(
      tsp, MGdata, split_config_MG,
      fewshot_fraction=1.0,
      fewshot_location=location
  )
  dset_test_sc =[]
  for i in range(906):
    data_test_sc = deepcopy(dset_test[i])
    data_test_sc['past_values'] = (dset_test[i]['past_values']-scale_mean[i])/scale_sd[i]
    data_test_sc['future_values'] = (dset_test[i]['future_values']-scale_mean[i])/scale_sd[i]
    dset_test_sc.append(data_test_sc)

  # model init
  zeroshot_trainer = Trainer(
      model=model,
      args=TrainingArguments(
          output_dir=temp_dir,
          per_device_eval_batch_size=64,
          seed = 0,
      ),
  )

  # first predction
  predictions_dict_first = zeroshot_trainer.predict(dset_test_sc)
  pred = predictions_dict_first.predictions[0]

  # rolling prediction


  preds = pred
  preds_pt = preds
  for i in range(16):
    dset_test_sc = update_past_values_with_predictions(dset_test_sc, pred)
    predictions_dict = zeroshot_trainer.predict(dset_test_sc)
    pred = predictions_dict.predictions[0]
    preds = np.concatenate((preds,pred),axis = 1)
    if i < 10:
      preds_pt = preds
    else:
      dset_test_sc_pt = perturb_last_element(dset_test_sc)
      predictions_dict_pt = zeroshot_trainer.predict(dset_test_sc_pt)
      pred_pt = predictions_dict_pt.predictions[0]
      preds_pt = np.concatenate((preds_pt,pred_pt),axis = 1)

  # squeeze dimension and reverse scale
  preds = preds.squeeze(-1)
  preds = preds*scale_sd[:,np.newaxis]+scale_mean[:,np.newaxis]
  preds_pt = preds_pt.squeeze(-1)
  preds_pt = preds_pt*scale_sd[:,np.newaxis]+scale_mean[:,np.newaxis]

  df.loc[len(df)-1,'Pred'] = [preds]
  df.loc[len(df)-1,'Pred_pt'] = [preds_pt]

In [161]:
df = pd.DataFrame(columns=['True','Pred','Pred_pt','tau','frac','location'])
for tau in [60,120,200]:
  for frac in [5,30,75]:
    for location in ["last","uniform"]:
      rolling_prediction(tau,frac,location)

In [169]:
df.to_pickle("/content/drive/MyDrive/rolling_prediction.pkl")

In [162]:
df.to_csv("/content/drive/MyDrive/rolling_prediction.csv")